# Regularisation, Initialisation & Normalisation

Implementation of fundamental techniques for improving the training and generalisation of neural networks.

This notebook covers regularisation with inverted dropout, Xavier (Glorot) parameter initialisation, and batch normalisation.

In this assignment, the goal is to get familiar with some tools that can help to speed up the training process of neural networks. **Regularisation** is a technique that can be used to avoid overfitting. Knowing what kind of **initialisation** to use in what context is often important to assure fast learning. **Normalisation** is a tool that tackles the problem of drifting distributions that pops up in very deep networks and hinders learning.

In [ ]:
import numpy as np

from nnumpy import Module, Parameter
from nnumpy.testing import gradient_check

rng = np.random.default_rng(1856)

In [ ]:
def initialiser(fn):
    """ 
    Function decorator for initialisation functions that
    enables initialisation of multiple weight arrays at once. 
    """
    
    def init_wrapper(*parameters, **kwargs):
        for par in parameters:
            par[:] = fn(par.shape, **kwargs)
            par.zero_grad()
    
    init_wrapper.__name__ = fn.__name__ + "_init"
    init_wrapper.__doc__ = fn.__doc__
    return init_wrapper

## Regularisation

Neural networks are infamously prone to overfitting. Just as with any machine learning model, overfitting can relatively easily be detected by monitoring the learning curves on training and validation sets. In order to counter these effects, you can use regularisation techniques. 

![learning curves](https://d2l.ai/_images/capacity-vs-error.svg)

One possibility is to use well-known approaches from regression: e.g. $L_1$ or $L_2$ regularisation, which are also known as *LASSO*, resp. *ridge* regression. Also simply interrupting the learning before the overfitting occurs can prevent overfitting models. These are only a few examples, but most regularisation techniques are not exlusive to neural networks. However, there is one NN-exclusive approach that is very commonly used: **Dropout**.

### Inverted Dropout ()

Dropout is a simple, but very effective regularisation technique that can be added practically anywhere in a network. The idea of dropout is to randomly disable a few neurons during training. During inference all neurons are used. Since this would lead to a shift in distribution of the pre-activations in the next layer (training vs inference), the neurons are scaled down during evaluation so that the distributions during inference and training are approximately the same. In order to avoid the need to change the network during evaluation, it is also possible to scale up the activations during training. This specific change in implementation is often referred to as *inverted* dropout.

> Implement the forward and backward pass of an **inverted dropout** module.

**Hint:** use the `Module` attribute `predicting` to check whether you are in prediction or training or mode.

In [ ]:
class Dropout(Module):
    """ NNumpy implementation of (inverted) dropout. """

    def __init__(self, rate: float = .5, seed: int = None):
        """
        Parameters
        ----------
        rate : float, optional
            The percentage of neurons to be dropped.
        seed : int, optional
            Seed for the pseudo random generator.
        """
        super().__init__()
        if rate < 0. or rate > 1.:
            raise ValueError("dropout rate should be between zero and one")

        self.rate = float(rate)
        self.rng = np.random.default_rng(seed)

    def compute_outputs(self, x):
        # In prediction mode: do nothing
        if self.predicting or self.rate == 0.0:
            return x, None

        # Training mode: inverted dropout
        keep_prob = 1.0 - self.rate
        mask = self.rng.random(x.shape) < keep_prob
        out = x * mask / keep_prob

        return out, mask


    def compute_grads(self, grads, cache):
         # If no dropout was applied
        if cache is None:
            return grads

        mask = cache
        keep_prob = 1.0 - self.rate

        return grads * mask / keep_prob

In [ ]:
# Test Cell: do not edit or delete!
dropout_layer = Dropout(rate=0.5)
x = np.ones(7)
y, cache = dropout_layer.compute_outputs(x)
assert isinstance(y, np.ndarray), (
    "ex1: the output of Dropout.compute_outputs is not a numpy array"
)
assert y.shape == x.shape, (
    "ex1: the output of Dropout.compute_outputs has incorrect shape"
)

In [ ]:
# Test Cell: do not edit or delete!

In [ ]:
# Test Cell: do not edit or delete!

In [ ]:
# Test Cell: do not edit or delete!
dropout_layer = Dropout(rate=0.2)
x = rng.standard_normal(size=(1, 11, 13))
assert gradient_check(dropout_layer, x, debug=True), (
    "ex1: gradient check for Dropout failed"
)

## Initialisation

A good initialisation has proven to be very important to learn deep neural networks. Although this can be considered as a well-known fact, it is astonishing how often initialisation is ignored. Since simply initialising all parameters with some constant does not work, the initial values are generally small, randomly generated numbers. There are different distributions to sample these values from, however.

### Xavier Glorot Initialisation ()

When generating random values, there are different choices for the distribution to draw numbers from. The uniform or Gaussian (a.k.a. normal) distributions are most common for initialising the parameters of a neural network. After all, these are simple distributions that can easily be centred around zero.

Apart from centring the initial parameters around zero, it is also helpful to make sure that the weights have a specific amount of variance. Xavier Glorot proposed to use the reciprocal of the average of fan-in and fan-out, i.e. $\frac{2}{\text{fan-in} + \text{fan-out}}$, for the variance. Here, *fan-in* and *fan-out* are the number of incoming connections per output neuron and number of outgoing connections per input neuron, respectively.

Note, however, that this proposal only holds for identity and $\tanh$ activation functions. When using different activation functions, the variance of the initial parameters need to be scaled correspondingly. This can be done by means of a linear *gain* factor that accounts for the effect of the activation functions.

 > Implement the `glorot_uniform` function so that it produces initial weights for a parameter with given shape according to the proposal from Xavier Glorot. Make sure to make use of the seed for the initialisation, as well as the `gain` parameter.
 
**Hint:** Think carefully about the number of connections in convolutional layers.

In [ ]:
@initialiser
def glorot_uniform(shape, gain: float = 1., seed: int = None):
    """
    Initialise parameter cf. Glorot, using a uniform distribution.
    
    Parameters
    ----------
    shape : tuple
        The shape of the parameter to be initialised.
    gain : float, optional
        Multiplier for the variance of the initialisation.
    seed : int, optional
        Seed for generating pseudo random numbers.
        
    Returns
    -------
    values: ndarray
        Numpy array with the initial weight values
        with dimensions as specified by `shape`.
    """
    rng = np.random.default_rng(seed)

    if len(shape) == 2:
        # Dense layer
        fan_in, fan_out = shape

    else:
        # Convolutional layer
        # shape = (out_channels, in_channels, *kernel_size)
        fan_out = shape[0]
        fan_in = shape[1]
        kernel_size = np.prod(shape[2:])

        fan_in *= kernel_size
        fan_out *= kernel_size

    limit = gain * np.sqrt(6.0 / (fan_in + fan_out))
    return rng.uniform(-limit, limit, size=shape)

In [ ]:
# Test Cell: do not edit or delete!
par = Parameter(np.ones((10, 10)))
glorot_uniform(par, seed=1806)
assert np.all(par != 1), (
    "ex2: glorot_uniform function does not return (reasonable) values"
)

In [ ]:
# Test Cell: do not edit or delete!
par1 = Parameter(np.ones((10, 10)))
par2 = Parameter(np.ones((100, 100)))
glorot_uniform(par1, par2, seed=1806)
assert not np.isclose(par1.var(), par2.var(), atol=1e-3), (
    "ex2: glorot_uniform function does not (correctly) use FC parameter shape"
)
par1 = Parameter(np.ones((10, 10, 3, 3)))
par2 = Parameter(np.ones((100, 100, 3, 3)))
glorot_uniform(par1, par2, seed=1806)
assert not np.isclose(par1.var(), par2.var(), atol=1e-3), (
    "ex2: glorot_uniform function does not (correctly) use Conv parameter shape"
)

In [ ]:
# Test Cell: do not edit or delete!

In [ ]:
# Test Cell: do not edit or delete!

## Normalisation

Whereas initialisation ensures proper variance propagation through the network when learning starts, it does not ensure that the weights keep these properties after some updates. To ensure a steady flow of information through the network, normalisation techniques were introduced. The idea of normalisation is to normalise either the activations or pre-activations. This can be done explicitly, using techniques like *Batch* or *Layer Normalisation*, or more implicitly, e.g. *weight normalisation* or using *self-normalising networks*.

### Batch Normalisation ()

Batch Normalisation (or *batch norm* for short) has empirically proven to be a very useful technique for improving the performance of neural networks. It is not quite clear why it works so well, but there is some form of consensus that it acts as a regulariser and improves gradient flow in the network. 

The core principle of batch norm is to subtract the mean and divide by the standard deviation of the data, computed over the samples in one batch. Each neuron is normalised individually, so that all neurons have zero mean and unit variance. Batch norm also uses parameters $\gamma$ and $\beta$ to scale, resp. shift the normalised signal. Note that, since batch norm relies on batch statistics, it requires a large batch size to work properly!

During inference, it is not uncommon to want a prediction for a single sample. Therefore, you generally do not want to use the mean computed during inference. Therefore, batch norm tracks the statistics of the data during training using a [moving average](https://en.wikipedia.org/wiki/Moving_average). During evaluation, these tracked statistics, i.e. the statistics of the training data, are used to normalise the previously unseen samples.

 > Implement the forward and backward pass of the batch normalisation module. Use a simple moving average for tracking the statistics.
 
**Hint:** You can track the statistics in attributes of the module.

In [ ]:
class BatchNormalisation(Module):
    """ NNumpy implementation of batch normalisation. """

    def __init__(self, dims: tuple, eps: float = 1e-8):
        """
        Parameters
        ----------
        dims : tuple of ints
            The shape of the incoming signal (without batch dimension).
        eps : float, optional
            Small value for numerical stability.
        """
        super().__init__()
        self.dims = tuple(dims)
        self.eps = float(eps)
        
        self.gamma = self.register_parameter('gamma', np.ones(self.dims))
        self.beta = self.register_parameter('beta', np.zeros(self.dims))
        
        self.running_count = 0
        self.running_stats = np.zeros((2, ) + self.dims)

    def compute_outputs(self, x):
        axes = (0,)

        if not self.predicting:
        # batch statistics
            mean = x.mean(axis=axes)
            var = x.var(axis=axes)

        # update running statistics (simple moving average)
            self.running_stats[0] = (
                self.running_stats[0] * self.running_count + mean
            ) / (self.running_count + 1)

            self.running_stats[1] = (
                self.running_stats[1] * self.running_count + var
            ) / (self.running_count + 1)

            self.running_count += 1
        else:
        # use running statistics
            mean = self.running_stats[0]
            var = self.running_stats[1]

        std = np.sqrt(var + self.eps)
        x_hat = (x - mean) / std
        out = self.gamma * x_hat + self.beta

        cache = (x_hat, std)
        return out, cache

    def compute_grads(self, grads, cache):
        x_hat, std = cache
        N = grads.shape[0]

        self.beta.grad += grads.sum(axis=0)
        self.gamma.grad += (grads * x_hat).sum(axis=0)

        dx_hat = grads * self.gamma
        dx = (1.0 / N) * (1.0 / std) * (
            N * dx_hat
            - dx_hat.sum(axis=0)
            - x_hat * (dx_hat * x_hat).sum(axis=0)
        )

        return dx

In [ ]:
# Test Cell: do not edit or delete!
x = np.linspace(-1, 3, 50).reshape(10, 5)
bn = BatchNormalisation(x.shape[1:])
y_train, _ = bn.compute_outputs(x)
assert isinstance(y_train, np.ndarray), (
    "ex3: output of BatchNormalisation.compute_outputs is not a numpy array"
)
assert y_train.shape == x.shape, (
    "ex3: output of BatchNormalisation.compute_outputs has incorrect shape"
)

bn.eval()
y_eval, _ = bn.compute_outputs(x)
assert isinstance(y_eval, np.ndarray), (
    "ex3: output of BatchNormalisation.compute_outputs is not a numpy array in eval mode"
)
assert y_eval.shape == x.shape, (
    "ex3: output of BatchNormalisation.compute_outputs has incorrect shape in eval mode"
)

In [ ]:
# Test Cell: do not edit or delete!
x = np.linspace(-1, 3, 50).reshape(10, 5)
bn = BatchNormalisation(x.shape[1:])
y_train, _ = bn.compute_outputs(x)
assert np.isclose(y_train.mean(), 0.), (
    "ex3: BatchNormalisation.compute_outputs does not produce zero-mean outputs"
)
assert np.isclose(y_train.var(), 1.), (
    "ex3: BatchNormalisation.compute_outputs does not produce unit variance outputs"
)

bn.eval()
y_eval, _ = bn.compute_outputs(x)
assert np.isclose(y_eval.mean(), 0.), (
    "ex3: BatchNormalisation.compute_outputs does not produce zero-mean outputs in eval mode"
)
assert np.isclose(y_eval.var(), 1.), (
    "ex3: BatchNormalisation.compute_outputs does not produce unit variance outputs in eval mode"
)

In [ ]:
# Test Cell: do not edit or delete!

In [ ]:
# Test Cell: do not edit or delete!

In [ ]:
# Test Cell: do not edit or delete!
shape_ = (7, 3, 5)
x = rng.uniform(0, 3, size= shape_)
bn = BatchNormalisation(x.shape[1:])
assert gradient_check(bn, x, debug=True), (
    "ex3: gradient check for BatchNormalization failed"
)